# Notebook 1: Data Loading and Cleaning

---

## Objective

This notebook loads the raw Online Retail dataset and performs initial data cleaning to prepare it for analysis. Raw transactional data often contains missing values, cancellations, and invalid entries that must be handled before analysis.

---

## Business Context

The dataset contains all transactions from a UK-based online retailer between December 2010 and December 2011. The company sells unique all-occasion gifts, with many customers being wholesalers.

Before we can analyze customer behavior or sales patterns, we must:

1. Remove transactions with missing customer IDs
2. Exclude cancelled orders
3. Remove invalid quantities (zero or negative)
4. Remove invalid prices (zero or negative)
5. Filter out non-product stock codes (shipping charges, bank fees, etc.)

---

## What This Notebook Covers

- Loading the Excel dataset from raw data folder
- Initial data inspection (shape, columns, data types)
- Handling missing values
- Removing cancelled transactions
- Filtering invalid quantities and prices
- Creating TotalPrice column for analysis
- Saving cleaned data for subsequent notebooks

---

## Expected Output

A cleaned dataset saved to `../data/processed/online_retail_clean.csv`

---

## Begin Analysis

Run the cells below to load and clean the data.


### Loading and Cleaning the data

#### Importing the Libraries 

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load the dataset from your local raw data folder
# Update the filename if your file has a different name
file_path = '../data/raw/Online Retail.xlsx'
df = pd.read_excel(file_path)

In [5]:
print("Dataset shape:", df.shape)
print('-'*140)
print("\nFirst 5 rows:")
df.head()
print('-'*140)

print("\nDataset info:")
print(df.info())
print('-'*140)

print("\nMissing values:")
print(df.isnull().sum())
print('-'*140)

print("\nStatistical summary:")
print(df.describe())
print('-'*140)

Dataset shape: (541909, 8)
--------------------------------------------------------------------------------------------------------------------------------------------

First 5 rows:
--------------------------------------------------------------------------------------------------------------------------------------------

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), st

#### Data Cleaning

In [6]:
print("Original shape:", df.shape)

# Remove rows with missing CustomerID
df_clean = df.dropna(subset=['CustomerID'])
print(f"After removing missing CustomerID: {df_clean.shape}")

# Remove cancelled transactions (InvoiceNo starting with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancellations: {df_clean.shape}")

# Remove rows with zero or negative quantity
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"After removing non-positive quantity: {df_clean.shape}")

# Remove rows with zero or negative unit price
df_clean = df_clean[df_clean['UnitPrice'] > 0]
print(f"After removing non-positive price: {df_clean.shape}")

# Remove stock codes that are not actual products
non_products = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'TEST', 'AMAZONFEE', 'C2', 'DOT']
df_clean = df_clean[~df_clean['StockCode'].isin(non_products)]
print(f"After removing non-product codes: {df_clean.shape}")

# Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Convert InvoiceDate to datetime if not already
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Extract date components for analysis
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek
df_clean['WeekdayName'] = df_clean['InvoiceDate'].dt.day_name()

print("\nFinal cleaned dataset shape:", df_clean.shape)
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

# Save cleaned data to processed folder
df_clean.to_csv('../data/processed/online_retail_clean.csv', index=False)
print("\nCleaned data saved to ../data/processed/online_retail_clean.csv")

Original shape: (541909, 8)
After removing missing CustomerID: (406829, 8)
After removing cancellations: (397924, 8)
After removing non-positive quantity: (397924, 8)
After removing non-positive price: (397884, 8)
After removing non-product codes: (396337, 8)

Final cleaned dataset shape: (396337, 15)

Missing values after cleaning:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
Year           0
Month          0
Day            0
Hour           0
DayOfWeek      0
WeekdayName    0
dtype: int64

Cleaned data saved to ../data/processed/online_retail_clean.csv
